# Credit Risk & Default Analytics

## Dataset

This portfolio project uses the publicly available UCI Default of Credit Card Clients dataset.

The dataset contains historical credit-limit, repayment-status, billing, payment, and demographic attributes for 30,000 credit-card clients.

The target variable indicates whether a client defaulted on payment in the following month.

This project is developed solely for educational and portfolio purposes using publicly available data.

## Objective

The goal is to demonstrate practical skills in:

- Credit-risk analytics
- Exploratory data analysis
- SQL analysis
- Feature engineering
- Machine learning
- Probability-of-default style analysis
- Model evaluation
- Business interpretation
- Interactive dashboard development

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_FILE = PROJECT_ROOT / "data" / "raw" / "uci_credit_default_raw.csv"

credit_df = pd.read_csv(RAW_DATA_FILE)

print("Dataset shape:", credit_df.shape)

credit_df.head()

Dataset shape: (30000, 25)


,ID,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,default_next_month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [2]:
print("Dataset columns:")
print(credit_df.columns.tolist())

Dataset columns:
['ID', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20', 'X21', 'X22', 'X23', 'default_next_month']


In [3]:
DICTIONARY_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "uci_variable_dictionary.csv"
)

dictionary_df = pd.read_csv(DICTIONARY_FILE)

print("Dictionary shape:", dictionary_df.shape)

dictionary_df

Dictionary shape: (25, 7)


,name,role,type,demographic,description,units,missing_values
0,ID,ID,Integer,NaN,NaN,NaN,no
1,X1,Feature,Integer,NaN,LIMIT_BAL,NaN,no
2,X2,Feature,Integer,Sex,SEX,NaN,no
3,X3,Feature,Integer,Education Level,EDUCATION,NaN,no
4,X4,Feature,Integer,Marital Status,MARRIAGE,NaN,no
5,X5,Feature,Integer,Age,AGE,NaN,no
6,X6,Feature,Integer,NaN,PAY_0,NaN,no
7,X7,Feature,Integer,NaN,PAY_2,NaN,no
8,X8,Feature,Integer,NaN,PAY_3,NaN,no
9,X9,Feature,Integer,NaN,PAY_4,NaN,no


In [4]:
# Show the full UCI variable dictionary
pd.set_option("display.max_rows", None)

dictionary_df[
    [
        "name",
        "role",
        "type",
        "demographic",
        "description",
    ]
]

,name,role,type,demographic,description
0,ID,ID,Integer,NaN,NaN
1,X1,Feature,Integer,NaN,LIMIT_BAL
2,X2,Feature,Integer,Sex,SEX
3,X3,Feature,Integer,Education Level,EDUCATION
4,X4,Feature,Integer,Marital Status,MARRIAGE
5,X5,Feature,Integer,Age,AGE
6,X6,Feature,Integer,NaN,PAY_0
7,X7,Feature,Integer,NaN,PAY_2
8,X8,Feature,Integer,NaN,PAY_3
9,X9,Feature,Integer,NaN,PAY_4


In [5]:
# Inspect important categorical and target values

columns_to_inspect = [
    "X2",   # Sex
    "X3",   # Education
    "X4",   # Marital status
    "X6",   # Repayment status
    "X7",
    "X8",
    "X9",
    "X10",
    "X11",
    "default_next_month",
]

for column in columns_to_inspect:
    print(
        column,
        sorted(
            credit_df[column]
            .dropna()
            .unique()
            .tolist()
        )
    )

X2 [1, 2]
X3 [0, 1, 2, 3, 4, 5, 6]
X4 [0, 1, 2, 3]
X6 [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
X7 [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
X8 [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
X9 [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
X10 [-2, -1, 0, 2, 3, 4, 5, 6, 7, 8]
X11 [-2, -1, 0, 2, 3, 4, 5, 6, 7, 8]
default_next_month [0, 1]


In [6]:
target_summary = (
    credit_df["default_next_month"]
    .value_counts()
    .sort_index()
    .rename_axis("default_next_month")
    .reset_index(name="record_count")
)

target_summary["percentage"] = (
    target_summary["record_count"]
    / len(credit_df)
    * 100
).round(2)

target_summary

,default_next_month,record_count,percentage
0,0,23364,77.88
1,1,6636,22.12


## 2. Exploratory Credit Risk Analysis

This section analyzes the portfolio population, target distribution, borrower characteristics, credit limits, and repayment behavior to identify patterns associated with default risk.

In [7]:
PROCESSED_DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "credit_default_processed.csv"
)

processed_df = pd.read_csv(PROCESSED_DATA_FILE)

print("Processed dataset shape:", processed_df.shape)

processed_df.head()

Processed dataset shape: (30000, 28)


,customer_id,credit_limit,sex,education,marital_status,age,default_next_month,sex_code,education_code,marital_status_code,...,bill_amount_jul,bill_amount_jun,bill_amount_may,bill_amount_apr,payment_amount_sep,payment_amount_aug,payment_amount_jul,payment_amount_jun,payment_amount_may,payment_amount_apr
0,1,20000,Female,University,Married,24,1,2,2,1,...,689,0,0,0,0,689,0,0,0,0
1,2,120000,Female,University,Single,26,1,2,2,2,...,2682,3272,3455,3261,0,1000,1000,1000,0,2000
2,3,90000,Female,University,Single,34,0,2,2,2,...,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000
3,4,50000,Female,University,Married,37,0,2,2,1,...,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000
4,5,50000,Male,University,Married,57,0,1,2,1,...,35835,20940,19146,19131,2000,36681,10000,9000,689,679


In [8]:
profile_summary = pd.DataFrame({
    "data_type": processed_df.dtypes,
    "missing_count": processed_df.isna().sum(),
    "unique_values": processed_df.nunique(),
})

profile_summary

,data_type,missing_count,unique_values
customer_id,int64,0,30000
credit_limit,int64,0,81
sex,str,0,2
education,str,0,5
marital_status,str,0,4
age,int64,0,56
default_next_month,int64,0,2
sex_code,int64,0,2
education_code,int64,0,7
marital_status_code,int64,0,4


In [9]:
default_distribution = (
    processed_df["default_next_month"]
    .value_counts()
    .sort_index()
    .rename_axis("default_status")
    .reset_index(name="record_count")
)

default_distribution["default_status"] = (
    default_distribution["default_status"]
    .map({
        0: "No Default",
        1: "Default",
    })
)

default_distribution["percentage"] = (
    default_distribution["record_count"]
    / len(processed_df)
    * 100
).round(2)

default_distribution

,default_status,record_count,percentage
0,No Default,23364,77.88
1,Default,6636,22.12


In [10]:
fig = px.bar(
    default_distribution,
    x="default_status",
    y="record_count",
    text="percentage",
    title="Portfolio Default Distribution",
    labels={
        "default_status": "Default Status",
        "record_count": "Number of Clients",
    },
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
)

fig.show()

### Default Risk by Credit Limit

Credit limits are segmented into bands to examine whether default behavior differs across different levels of available credit.

The analysis describes associations in the historical dataset and should not be interpreted as a causal relationship.

In [11]:
credit_limit_analysis = processed_df.copy()

credit_limit_analysis["credit_limit_band"] = pd.cut(
    credit_limit_analysis["credit_limit"],
    bins=[
        0,
        50000,
        100000,
        200000,
        300000,
        500000,
        float("inf"),
    ],
    labels=[
        "Under 50K",
        "50K–99K",
        "100K–199K",
        "200K–299K",
        "300K–499K",
        "500K+",
    ],
    right=False,
)

credit_limit_summary = (
    credit_limit_analysis
    .groupby(
        "credit_limit_band",
        observed=True,
    )
    .agg(
        clients=("customer_id", "count"),
        defaults=("default_next_month", "sum"),
        default_rate=("default_next_month", "mean"),
    )
    .reset_index()
)

credit_limit_summary["default_rate_pct"] = (
    credit_limit_summary["default_rate"] * 100
).round(2)

credit_limit_summary

,credit_limit_band,clients,defaults,default_rate,default_rate_pct
0,Under 50K,4311,1555,0.360705,36.07
1,50K–99K,7139,1857,0.260120,26.01
2,100K–199K,7400,1537,0.207703,20.77
3,200K–299K,6033,987,0.163600,16.36
4,300K–499K,4189,596,0.142277,14.23
5,500K+,928,104,0.112069,11.21


In [12]:
fig = px.bar(
    credit_limit_summary,
    x="credit_limit_band",
    y="default_rate_pct",
    text="default_rate_pct",
    title="Default Rate by Credit Limit Band",
    labels={
        "credit_limit_band": "Credit Limit Band",
        "default_rate_pct": "Default Rate (%)",
    },
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
)

fig.update_layout(
    yaxis_range=[
        0,
        credit_limit_summary["default_rate_pct"].max() * 1.15,
    ]
)

fig.show()

### Key Finding

Default rates decline substantially as credit limits increase in this historical portfolio.

Clients with credit limits below 50K show a default rate of approximately 36%, compared with approximately 11% among clients with limits of 500K or more.

This relationship should be interpreted as an association rather than a causal effect. Credit limits may reflect other borrower characteristics and prior underwriting decisions that are also related to credit risk.

### Default Risk by Age

Borrowers are segmented into age bands to evaluate whether observed default behavior varies across different age groups.

In [13]:
age_analysis = processed_df.copy()

age_analysis["age_band"] = pd.cut(
    age_analysis["age"],
    bins=[
        0,
        25,
        35,
        45,
        55,
        65,
        float("inf"),
    ],
    labels=[
        "Under 25",
        "25–34",
        "35–44",
        "45–54",
        "55–64",
        "65+",
    ],
    right=False,
)

age_summary = (
    age_analysis
    .groupby(
        "age_band",
        observed=True,
    )
    .agg(
        clients=("customer_id", "count"),
        defaults=("default_next_month", "sum"),
        default_rate=("default_next_month", "mean"),
    )
    .reset_index()
)

age_summary["default_rate_pct"] = (
    age_summary["default_rate"] * 100
).round(2)

age_summary

,age_band,clients,defaults,default_rate,default_rate_pct
0,Under 25,2685,730,0.271881,27.19
1,25–34,13011,2641,0.202982,20.30
2,35–44,9018,1971,0.218563,21.86
3,45–54,4233,1013,0.239310,23.93
4,55–64,943,253,0.268293,26.83
5,65+,110,28,0.254545,25.45


In [14]:
fig = px.bar(
    age_summary,
    x="age_band",
    y="default_rate_pct",
    text="default_rate_pct",
    title="Default Rate by Age Group",
    labels={
        "age_band": "Age Group",
        "default_rate_pct": "Default Rate (%)",
    },
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
)

fig.update_layout(
    yaxis_range=[
        0,
        age_summary["default_rate_pct"].max() * 1.15,
    ]
)

fig.show()

### Key Finding

Default risk varies across age groups but does not follow a simple linear pattern.

Clients aged 25–34 show the lowest observed default rate at approximately 20%, while the youngest and older age groups exhibit higher rates.

Age alone therefore appears to provide limited risk differentiation compared with stronger behavioral indicators. Segment size and other borrower characteristics should also be considered before drawing conclusions.

### Default Risk by Recent Repayment Status

Recent repayment status is examined to determine how historical payment behavior relates to subsequent default risk.

In [15]:
repayment_summary = (
    processed_df
    .groupby("repayment_status_sep")
    .agg(
        clients=("customer_id", "count"),
        defaults=("default_next_month", "sum"),
        default_rate=("default_next_month", "mean"),
    )
    .reset_index()
    .sort_values("repayment_status_sep")
)

repayment_summary["default_rate_pct"] = (
    repayment_summary["default_rate"] * 100
).round(2)

repayment_summary

,repayment_status_sep,clients,defaults,default_rate,default_rate_pct
0,-2,2759,365,0.132294,13.23
1,-1,5686,954,0.167781,16.78
2,0,14737,1888,0.128113,12.81
3,1,3688,1252,0.339479,33.95
4,2,2667,1844,0.691414,69.14
5,3,322,244,0.757764,75.78
6,4,76,52,0.684211,68.42
7,5,26,13,0.500000,50.00
8,6,11,6,0.545455,54.55
9,7,9,7,0.777778,77.78


In [16]:
fig = px.bar(
    repayment_summary,
    x="repayment_status_sep",
    y="default_rate_pct",
    text="default_rate_pct",
    title="Default Rate by Most Recent Repayment Status",
    labels={
        "repayment_status_sep": "Repayment Status",
        "default_rate_pct": "Default Rate (%)",
    },
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
)

fig.update_layout(
    yaxis_range=[
        0,
        repayment_summary["default_rate_pct"].max() * 1.15,
    ]
)

fig.show()

In [17]:
repayment_summary[
    [
        "repayment_status_sep",
        "clients",
        "defaults",
        "default_rate_pct",
    ]
]

,repayment_status_sep,clients,defaults,default_rate_pct
0,-2,2759,365,13.23
1,-1,5686,954,16.78
2,0,14737,1888,12.81
3,1,3688,1252,33.95
4,2,2667,1844,69.14
5,3,322,244,75.78
6,4,76,52,68.42
7,5,26,13,50.00
8,6,11,6,54.55
9,7,9,7,77.78


In [18]:
repayment_band_df = processed_df.copy()

def repayment_band(status):
    if status <= 0:
        return "No Positive Delinquency Code"
    elif status == 1:
        return "1-Month Delay"
    elif status == 2:
        return "2-Month Delay"
    else:
        return "3+ Month Delay"

repayment_band_df["repayment_risk_band"] = (
    repayment_band_df["repayment_status_sep"]
    .apply(repayment_band)
)

repayment_band_summary = (
    repayment_band_df
    .groupby("repayment_risk_band")
    .agg(
        clients=("customer_id", "count"),
        defaults=("default_next_month", "sum"),
        default_rate=("default_next_month", "mean"),
    )
    .reset_index()
)

repayment_band_summary["default_rate_pct"] = (
    repayment_band_summary["default_rate"] * 100
).round(2)

repayment_band_summary

,repayment_risk_band,clients,defaults,default_rate,default_rate_pct
0,1-Month Delay,3688,1252,0.339479,33.95
1,2-Month Delay,2667,1844,0.691414,69.14
2,3+ Month Delay,463,333,0.719222,71.92
3,No Positive Delinquency Code,23182,3207,0.138340,13.83


In [19]:
band_order = [
    "No Positive Delinquency Code",
    "1-Month Delay",
    "2-Month Delay",
    "3+ Month Delay",
]

repayment_band_summary["repayment_risk_band"] = pd.Categorical(
    repayment_band_summary["repayment_risk_band"],
    categories=band_order,
    ordered=True,
)

repayment_band_summary = (
    repayment_band_summary
    .sort_values("repayment_risk_band")
)

fig = px.bar(
    repayment_band_summary,
    x="repayment_risk_band",
    y="default_rate_pct",
    text="default_rate_pct",
    title="Default Rate by Recent Delinquency Severity",
    labels={
        "repayment_risk_band": "Recent Repayment Behavior",
        "default_rate_pct": "Default Rate (%)",
    },
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
)

fig.update_layout(
    yaxis_range=[
        0,
        repayment_band_summary["default_rate_pct"].max() * 1.15,
    ]
)

fig.show()

### Key Finding

Recent repayment behavior shows a strong relationship with subsequent default risk.

Clients with no positive delinquency code have an observed default rate of approximately 14%, compared with approximately 34% for clients with a one-month delay and about 69%–72% for clients with more severe delinquency.

This suggests that recent repayment behavior is likely to be one of the most informative predictors in the credit-risk model. The relationship represents historical association and should not be interpreted as causal.

### Default Risk by Repeated Delinquency

A borrower who has been delinquent repeatedly may represent greater risk than a borrower with a single isolated delay.

This analysis counts the number of months during the six-month observation period in which the repayment-status code was positive.

In [20]:
repayment_columns = [
    "repayment_status_sep",
    "repayment_status_aug",
    "repayment_status_jul",
    "repayment_status_jun",
    "repayment_status_may",
    "repayment_status_apr",
]

delinquency_df = processed_df.copy()

delinquency_df["delinquent_months"] = (
    delinquency_df[repayment_columns]
    .gt(0)
    .sum(axis=1)
)

delinquency_summary = (
    delinquency_df
    .groupby("delinquent_months")
    .agg(
        clients=("customer_id", "count"),
        defaults=("default_next_month", "sum"),
        default_rate=("default_next_month", "mean"),
    )
    .reset_index()
)

delinquency_summary["default_rate_pct"] = (
    delinquency_summary["default_rate"] * 100
).round(2)

delinquency_summary

,delinquent_months,clients,defaults,default_rate,default_rate_pct
0,0,19931,2334,0.117104,11.71
1,1,4426,1320,0.298238,29.82
2,2,1899,736,0.387572,38.76
3,3,1154,587,0.508666,50.87
4,4,951,545,0.573081,57.31
5,5,298,171,0.573826,57.38
6,6,1341,943,0.703207,70.32


In [21]:
fig = px.bar(
    delinquency_summary,
    x="delinquent_months",
    y="default_rate_pct",
    text="default_rate_pct",
    title="Default Rate by Number of Delinquent Months",
    labels={
        "delinquent_months": "Number of Delinquent Months",
        "default_rate_pct": "Default Rate (%)",
    },
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
)

fig.update_layout(
    yaxis_range=[
        0,
        delinquency_summary["default_rate_pct"].max() * 1.15,
    ]
)

fig.show()

### Key Finding

Default risk rises substantially as delinquency becomes more persistent across the six-month observation period.

Clients with no delinquent months have an observed default rate of approximately 12%, compared with about 30% after one delinquent month, 51% after three delinquent months, and 70% among clients delinquent in all six observed months.

This suggests that delinquency frequency captures important behavioral risk information and may be a useful engineered feature for predictive modeling.

### Default Risk by Credit Utilization

Credit utilization compares the most recent billed balance with the client's available credit limit.

Higher utilization may indicate greater borrowing pressure, although utilization should be interpreted together with repayment behavior and other borrower characteristics.

In [22]:
utilization_df = processed_df.copy()

utilization_df["credit_utilization"] = (
    utilization_df["bill_amount_sep"]
    / utilization_df["credit_limit"]
)

utilization_df["utilization_band"] = pd.cut(
    utilization_df["credit_utilization"],
    bins=[
        float("-inf"),
        0,
        0.25,
        0.50,
        0.75,
        1.00,
        float("inf"),
    ],
    labels=[
        "0% or Less",
        "1%–25%",
        "26%–50%",
        "51%–75%",
        "76%–100%",
        "Over 100%",
    ],
    right=True,
)

utilization_summary = (
    utilization_df
    .groupby(
        "utilization_band",
        observed=True,
    )
    .agg(
        clients=("customer_id", "count"),
        defaults=("default_next_month", "sum"),
        default_rate=("default_next_month", "mean"),
    )
    .reset_index()
)

utilization_summary["default_rate_pct"] = (
    utilization_summary["default_rate"] * 100
).round(2)

utilization_summary

,utilization_band,clients,defaults,default_rate,default_rate_pct
0,0% or Less,2598,643,0.247498,24.75
1,1%–25%,11440,1923,0.168094,16.81
2,26%–50%,3588,780,0.217391,21.74
3,51%–75%,3579,929,0.259570,25.96
4,76%–100%,6680,1725,0.258234,25.82
5,Over 100%,2115,636,0.300709,30.07


In [23]:
fig = px.bar(
    utilization_summary,
    x="utilization_band",
    y="default_rate_pct",
    text="default_rate_pct",
    title="Default Rate by Credit Utilization",
    labels={
        "utilization_band": "Credit Utilization",
        "default_rate_pct": "Default Rate (%)",
    },
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
)

fig.update_layout(
    yaxis_range=[
        0,
        utilization_summary["default_rate_pct"].max() * 1.15,
    ]
)

fig.show()

### Key Finding

Default risk generally increases as credit utilization rises, with the highest observed default rate among clients whose billed balance exceeds their credit limit.

Clients in the 1%–25% utilization range show the lowest observed default rate at approximately 17%, while clients above 100% utilization show a default rate of approximately 30%.

The relationship is not perfectly linear, indicating that utilization should be considered together with repayment history, credit limits, payment behavior, and other risk indicators.